### Imports

In [67]:
import pandas as pd
import random
import re

### Utils

In [68]:
def inject_noise(text, noise_type="markup"):
    if noise_type == "markup":
        tags = ["<div>", "<b>", "<span>", "<p>"]
        tag = random.choice(tags)
        return f"{tag}{text}{tag.replace('<', '</')}"
    
    elif noise_type == "symbols":
        syms = ["!!!", "@@@", "***", "###"]
        s = random.choice(syms)
        return f"{s} {text} {s}"
    
    elif noise_type == "marketing":
        phrases = ["[BEST SELLER]", "[LIMITED TIME]", "FREE SHIPPING!", "100% GENUINE"]
        return f"{random.choice(phrases)} {text}"
    
    elif noise_type == "case":
        return "".join(c.upper() if random.random() > 0.5 else c.lower() for c in text)
    
    return text

# Example Usage:
clean_name = "Adidas Ultraboost"
print(inject_noise(clean_name, "markup"))    # <b>Adidas Ultraboost</b>
print(inject_noise(clean_name, "marketing")) # [LIMITED TIME] Adidas Ultraboost

<p>Adidas Ultraboost</p>
[BEST SELLER] Adidas Ultraboost


### Dataset Analysis

In [69]:
df = pd.read_csv("../data/01-raw/amazon.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1465 entries, 0 to 1464
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   product_id           1465 non-null   str  
 1   product_name         1465 non-null   str  
 2   category             1465 non-null   str  
 3   discounted_price     1465 non-null   str  
 4   actual_price         1465 non-null   str  
 5   discount_percentage  1465 non-null   str  
 6   rating               1465 non-null   str  
 7   rating_count         1463 non-null   str  
 8   about_product        1465 non-null   str  
 9   user_id              1465 non-null   str  
 10  user_name            1465 non-null   str  
 11  review_id            1465 non-null   str  
 12  review_title         1465 non-null   str  
 13  review_content       1465 non-null   str  
 14  img_link             1465 non-null   str  
 15  product_link         1465 non-null   str  
dtypes: str(16)
memory usage: 4.7 MB


In [70]:
df.head()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


In [71]:
review_content_length = df["product_name"].str.len()
review_content_words = df["product_name"].str.split().str.len()

"len", review_content_length.describe(), "words", review_content_words.describe()

('len',
 count    1465.000000
 mean      128.418430
 std        51.632492
 min        16.000000
 25%        85.000000
 50%       128.000000
 75%       175.000000
 max       485.000000
 Name: product_name, dtype: float64,
 'words',
 count    1465.000000
 mean       20.646416
 std         8.419859
 min         2.000000
 25%        14.000000
 50%        20.000000
 75%        27.000000
 max        81.000000
 Name: product_name, dtype: float64)

In [72]:
df["product_brand"] = df["product_name"].str.split().str[0]

df["product_brand"].value_counts().head(10)

product_brand
boAt            67
Samsung         36
AmazonBasics    33
Portronics      31
Ambrane         29
Redmi           26
Fire-Boltt      26
Bajaj           26
Amazon          25
Wayona          24
Name: count, dtype: int64

### Inject Noise

In [73]:
df["noisy_product_name"] = df["product_name"].apply(lambda x: inject_noise(x, "marketing"))
df["noisy_product_name"].head()

0    [LIMITED TIME] Wayona Nylon Braided USB to Lig...
1    [BEST SELLER] Ambrane Unbreakable 60W / 3A Fas...
2    FREE SHIPPING! Sounce Fast Phone Charging Cabl...
3    100% GENUINE boAt Deuce USB 300 2 in 1 Type-C ...
4    100% GENUINE Portronics Konnect L 1.2M Fast Ch...
Name: noisy_product_name, dtype: str

In [74]:
df_to_save = df[["product_id", "product_brand", "product_name", "noisy_product_name"]]

df_to_save.to_csv("../data/02-processed/features.csv", index=False)

In [75]:
df = df.sample(n=100, random_state=42).reset_index(drop=True)

In [76]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   product_id           100 non-null    str   
 1   product_name         100 non-null    str   
 2   category             100 non-null    str   
 3   discounted_price     100 non-null    str   
 4   actual_price         100 non-null    str   
 5   discount_percentage  100 non-null    str   
 6   rating               100 non-null    str   
 7   rating_count         100 non-null    str   
 8   about_product        100 non-null    str   
 9   user_id              100 non-null    str   
 10  user_name            100 non-null    str   
 11  review_id            100 non-null    str   
 12  review_title         100 non-null    str   
 13  review_content       100 non-null    str   
 14  img_link             100 non-null    str   
 15  product_link         100 non-null    str   
 16  product_brand       

In [77]:
df_to_save.to_csv("../data/03-results/features.csv", index=False)